In [1]:
# ==================================
#       1. Image acquisition
# ==================================
from google.colab import files
uploaded = files.upload()
IMAGE_PATH = "microscope1.jpg"  #MUST ADJUST

Saving microscope1.jpg to microscope1 (1).jpg


In [2]:
# ==================================
#       2. Environment Setup
# ==================================
import os

# Check for and display GPU information. This is crucial for performance with SAM.
!nvidia-smi

# Get the current working directory to create paths relative to the notebook.

HOME = os.getcwd()

# Install required libraries.
# `segment-anything` is the core library for the SAM model.
# `jupyter_bbox_widget` is for bounding box annotations (though not used in this specific script).
# `roboflow` and `dataclasses-json` are likely for dataset handling or metadata.
# `supervision` is a computer vision utility library used for annotations and detections.
!pip install -q 'git+https://github.com/facebookresearch/segment-anything.git'
!pip install -q jupyter_bbox_widget roboflow dataclasses-json supervision==0.23.0



# ==================================
#       3. Data and Model Download
# ==================================
import torch
import cv2
import supervision as sv
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

# Create a directory to store the model weights.
!mkdir -p {HOME}/weights

# Download the pre-trained SAM model checkpoint.
!wget -q https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -P {HOME}/weights

# Set the path to the downloaded model checkpoint and verify it exists.
CHECKPOINT_PATH = os.path.join(HOME, "weights", "sam_vit_b_01ec64.pth")
print(CHECKPOINT_PATH, "; exist:", os.path.isfile(CHECKPOINT_PATH))

/bin/bash: line 1: nvidia-smi: command not found
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.5/151.5 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/88.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 100.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 66.7 MB/s eta 0:00:00


In [5]:
# ==================================
#       4. Model Initialization
# ==================================

# Set the device for computation. It will use a CUDA-enabled GPU if available,
# otherwise it will fall back to the CPU (which is much slower).
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
MODEL_TYPE = "vit_b"

# Load the Segment Anything Model (SAM) with the specified checkpoint and device.
sam = sam_model_registry[MODEL_TYPE](checkpoint=CHECKPOINT_PATH).to(device=DEVICE)

# Initialize the automatic mask generator. This is the main component that
# will generate segmentation masks for the entire image without any prompts.

mask_generator = SamAutomaticMaskGenerator(
    model=sam,
    points_per_side=40, # Increase this value for more masks
    pred_iou_thresh=0.86,
    stability_score_thresh=0.92,
    crop_n_layers=1,
    crop_n_points_downscale_factor=2,
    min_mask_region_area=10000
)

# ==================================
#   5. Image Processing and Segmentation
# ==================================

# Read the image using OpenCV.
image_bgr = cv2.imread(IMAGE_PATH)

# Convert the image from BGR (OpenCV's default) to RGB, as SAM expects RGB input.
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

# Resize the image to a smaller resolution
max_dimension = 1024
height, width, _ = image_rgb.shape
if max(height, width) > max_dimension:
    scale = max_dimension / max(height, width)
    new_size = (int(width * scale), int(height * scale))
    image_rgb_resized = cv2.resize(image_rgb, new_size, interpolation=cv2.INTER_AREA)
else:
    image_rgb_resized = image_rgb

# Now, generate masks with the resized image
sam_result = mask_generator.generate(image_rgb_resized)

# Initialize the annotator from the `supervision` library.
# It will use a unique color for each segmented mask.
mask_annotator = sv.MaskAnnotator(color_lookup=sv.ColorLookup.INDEX)

# Convert the SAM results into `supervision.Detections` format.
# This structure is needed for easy annotation.
detections = sv.Detections.from_sam(sam_result=sam_result)

# Annotate the original image with the generated masks.
annotated_image = mask_annotator.annotate(scene=image_rgb_resized.copy(), detections=detections)

# ==================================
#       5. Visualization
# ==================================

# Display the original and the segmented image side by side using `supervision`.
sv.plot_images_grid(
    images=[image_bgr, annotated_image],
    grid_size=(1, 2),
    titles=['Source Image', 'Segmented Image']
)


/content/weights/sam_vit_b_01ec64.pth ; exist: True


KeyboardInterrupt: 

In [ ]:
# ==================================
#       6. Mask Visualization
# ==================================
import math
masks = [
    mask['segmentation']
    for mask
    in sorted(sam_result, key=lambda x: x['area'], reverse=True)
]

# Calculate the number of rows and columns
num_masks = len(masks)
n_cols = 8  # Fixed number of columns for a clean look
n_rows = math.ceil(num_masks / n_cols)  # Calculate rows, rounding up

sv.plot_images_grid(
    images=masks,
    grid_size=(n_rows, n_cols),
    size=(10, 3)
)


In [ ]:
# ==================================
#       7. Midpoint acquisition
# ==================================
import numpy as np
from skimage.morphology import skeletonize


# This is the skeletonization algorythm for the main tube
def get_skeleton(mask: np.ndarray) -> np.ndarray:
    """
    Computes the skeleton of a binary mask.

    The skeleton is a one-pixel-wide line that is equidistant
    from the boundaries of the mask.

    Args:
        mask (np.ndarray): A binary mask as a NumPy array.
                           Expected to be a boolean array or a 0/255 uint8 array.

    Returns:
        np.ndarray: The skeleton of the mask as a boolean NumPy array.
    """
    # Ensure the mask is a boolean array, as required by the skeletonize function
    boolean_mask = mask.astype(bool)

    # Compute the skeleton
    skeleton = skeletonize(boolean_mask)

    return skeleton


# This find the midle line of the gap between the electrodes
def find_midline_between_masks(mask1: np.ndarray, mask2: np.ndarray) -> np.ndarray:
    """
    Finds the skeletonized midline between two binary masks.

    Args:
        mask1 (np.ndarray): The first binary mask.
        mask2 (np.ndarray): The second binary mask.

    Returns:
        np.ndarray: A boolean mask representing the skeletonized midline.
    """
    # 1. Combine the two masks using a logical OR operation
    combined_mask = np.logical_or(mask1, mask2)

    # 2. Invert the combined mask to get the empty space (background)
    empty_space_mask = ~combined_mask

    # 3. Skeletonize the empty space to find the midline
    # The skeleton will be a single-pixel-wide line
    midline_skeleton = skeletonize(empty_space_mask)

    return midline_skeleton





# Mask of electrodes MUST COMPLETE
mask1 = masks[3]
mask2 = masks[5]

# Find the midline between mask1 and mask2
midline = find_midline_between_masks(mask1, mask2)

# Mask of tube MUST COMPLETE
sample_mask = masks[1]
skeleton_result = get_skeleton(sample_mask)

# Get dimensions from a mask
height, width = mask1.shape

# Create a blank image to draw on
combined_image = np.zeros((height, width, 3), dtype=np.uint8)


# Draw the midline in a distinct color
combined_image[midline] = [0, 255, 0]  # Green

# Draw the single mask's skeleton in another distinct color
combined_image[skeleton_result] = [255, 0, 0]

# Display the final image
plt.imshow(combined_image)
plt.title('Midline and Skeleton in a Single Image')
plt.axis('off')
plt.show()